# Step 7 — Pixel-based Anomaly Segmentation Baselines (ERFNet)

Single inference per image; simultaneous calculation of **MSP**, **MaxLogit**, and **MaxEntropy** from the logits, on all 5 validation datasets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Set the working directory to eval/
import os, sys

EVAL_DIR = '/content/drive/MyDrive/Comprehensive Road Scene Understanding for Autonomous Driving/MaskArchitectureAnomaly_CourseProject/eval'

assert os.path.isdir(EVAL_DIR), f'Folder not found: {EVAL_DIR}'
os.chdir(EVAL_DIR)
if EVAL_DIR not in sys.path:
    sys.path.insert(0, EVAL_DIR)

print('CWD:', os.getcwd())
print('erfnet.py presente:', os.path.exists('erfnet.py'))

In [ ]:
import glob, random
import numpy as np
import torch
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from sklearn.metrics import average_precision_score, roc_curve
from erfnet import ERFNet

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

NUM_CLASSES = 20
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

input_transform = Compose([
    Resize((512, 1024), Image.BILINEAR),
    ToTensor(),
])
target_transform = Compose([
    Resize((512, 1024), Image.NEAREST),
])


def fpr_at_95_tpr(scores, labels):
    """FPR when TPR >= 0.95. Replaces ood_metrics.fpr_at_95_tpr."""
    fpr, tpr, _ = roc_curve(labels, scores)
    idx = np.searchsorted(tpr, 0.95, side='left')
    idx = min(idx, len(fpr) - 1)
    return float(fpr[idx])

In [ ]:
# Load ERFNet pretrained
WEIGHTS_PATH = '../trained_models/erfnet_pretrained.pth'

def load_my_state_dict(model, state_dict):
    own = model.state_dict()
    for name, param in state_dict.items():
        key = name[len('module.'):] if name.startswith('module.') else name
        if key in own:
            own[key].copy_(param)
        else:
            print('NOT loaded:', name)
    return model

model = ERFNet(NUM_CLASSES)
state = torch.load(WEIGHTS_PATH, map_location='cpu')
model = load_my_state_dict(model, state).to(DEVICE).eval()
print('Model ERFNet load')

In [ ]:
# Post-hoc anomaly scoring functions, ground-truth remapping, and dataset evaluation.

def compute_anomaly_scores(logits_np):
    """ From logits [C,H,W] -> dict with MSP, MaxLogit, MaxEntropy (high = anomalous) """
    # MaxLogit
    maxlogit = -np.max(logits_np, axis=0)

    # softmax
    z = logits_np - np.max(logits_np, axis=0, keepdims=True)
    ez = np.exp(z)
    p = ez / np.sum(ez, axis=0, keepdims=True)

    # MSP
    msp = 1.0 - np.max(p, axis=0)

    # MaxEntropy: Shannon entropy of the softmax distribution, normalized to [0,1].
    eps = 1e-12
    entropy = -np.sum(p * np.log(p + eps), axis=0) / np.log(p.shape[0])

    return {'MSP': msp.astype(np.float32),
            'MaxLogit': maxlogit.astype(np.float32),
            'MaxEntropy': entropy.astype(np.float32)}


def remap_gt(gt, dataset_key):
    """Bring the GT to the common convention: 0=ind, 1=ood, 255=ignore"""
    g = gt.copy()
    if dataset_key == 'RoadAnomaly':
        g = np.where(g == 2, 1, g)

    return g


def get_gt_path(img_path):
    """The GTs are in labels_masks/ with the same numeric name but .png extension"""
    gt = img_path.replace('images', 'labels_masks')
    for ext in ('.webp', '.jpg', '.jpeg'):
        gt = gt.replace(ext, '.png')
    return gt


@torch.no_grad()
def evaluate_dataset(image_glob, dataset_key, display_name):
    """One forward pass per image, simultaneous calculation of the 3 scores """
    scores = {'MSP': [], 'MaxLogit': [], 'MaxEntropy': []}
    gts = []

    paths = sorted(glob.glob(os.path.expanduser(image_glob)))
    print(f'[{display_name}] {len(paths)} images found')
    if not paths:
        return None

    for i, p in enumerate(paths):
        img = input_transform(Image.open(p).convert('RGB')).unsqueeze(0).float().to(DEVICE)
        logits = model(img).squeeze(0).cpu().numpy()  # [C,H,W]
        s = compute_anomaly_scores(logits)

        gt = np.array(target_transform(Image.open(get_gt_path(p))))
        gt = remap_gt(gt, dataset_key)

        if 1 not in np.unique(gt):
            continue

        gts.append(gt)
        for k in scores:
            scores[k].append(s[k])

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(paths)}')

        del logits, s
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    if not gts:
        print(f"  Warning: No valid ground truth images found for {display_name} after filtering. Skipping evaluation.")
        return None

    gts_arr = np.array(gts)

    # Build a valid-pixel mask that excludes ignore-index pixels (255).
    valid_mask = (gts_arr != 255)

    # Extract binary ground-truth labels (0 = inlier, 1 = anomaly) for valid pixels.
    val_lab = gts_arr[valid_mask]

    results = {}
    for method, lst in scores.items():
        arr = np.array(lst)

        # Extract anomaly scores for valid pixels only.
        val_out = arr[valid_mask]
        auprc = average_precision_score(val_lab, val_out) * 100.0
        fpr = fpr_at_95_tpr(val_out, val_lab) * 100.0

        results[method] = (auprc, fpr)

    return results


In [ ]:
# Configure paths and settings for the 5 anomaly validation datasets.
VALIDATION_ROOT = '../Validation_Dataset'

DATASETS = {
    'SMIYC RA-21':  ('RoadAnomaly21',     f'{VALIDATION_ROOT}/RoadAnomaly21/images/*.png'),
    'SMIYC RO-21':  ('RoadObsticle21',    f'{VALIDATION_ROOT}/RoadObsticle21/images/*.webp'),
    'FS L&F':       ('FS_LostFound_full', f'{VALIDATION_ROOT}/FS_LostFound_full/images/*.png'),
    'FS Static':    ('fs_static',         f'{VALIDATION_ROOT}/fs_static/images/*.jpg'),
    'Road Anomaly': ('RoadAnomaly',       f'{VALIDATION_ROOT}/RoadAnomaly/images/*.jpg'),
}

# Sanity check: verify dataset paths and sample counts.
for name, (key, pat) in DATASETS.items():
    print(f'{name:12s}: {len(glob.glob(pat))} images')

In [ ]:
# Run anomaly evaluation across all 5 datasets and all 3 scoring methods.
all_results = {}
for display_name, (key, pat) in DATASETS.items():
    print(f'\n=== {display_name} ===')
    res = evaluate_dataset(pat, key, display_name)
    if res is None:
        continue
    all_results[display_name] = res
    for m, (a, f) in res.items():
        print(f'  {m:11s}  AuPRC={a:6.2f}   FPR95={f:6.2f}')

In [ ]:
# Results table: AuPRC and FPR95 per method and dataset.
import pandas as pd

METHODS = ['MSP', 'MaxLogit', 'MaxEntropy']
DS_ORDER = ['SMIYC RA-21', 'SMIYC RO-21', 'FS L&F', 'FS Static', 'Road Anomaly']

rows = []
for m in METHODS:
    row = {'Model': 'ERFNet', 'Method': m}
    for ds in DS_ORDER:
        if ds in all_results and m in all_results[ds]:
            a, f = all_results[ds][m]
            row[f'{ds} AuPRC'] = round(a, 2)
            row[f'{ds} FPR95'] = round(f, 2)
        else:
            row[f'{ds} AuPRC'] = None
            row[f'{ds} FPR95'] = None
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv('step7_results.csv', index=False)
print('Saved in: step7_results.csv\n')
df


## 9. ERFNet mIoU on Cityscapes (val)

Calculating the mean IoU on the Cityscapes validation set by reusing the model already loaded in step 4. No extraction and no generation of `_labelTrainIds.png`: images and ground truths (`_gtFine_labelIds.png`) are read in-place by the two official zips, and the labelId (0..33) are remapped to `trainId` (0..18) at runtime via `torchvision.datasets.Cityscapes.classes`. Same trick used in `eomt/datasets/cityscapes_semantic.py`.

In [ ]:
import io, zipfile
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Compose, Resize, ToTensor
from torchvision.datasets import Cityscapes as CSMeta
from PIL import Image

from iouEval import iouEval, getColorEntry


CITYSCAPES_DIR = '/content/drive/MyDrive/Comprehensive Road Scene Understanding for Autonomous Driving/datasets/cityscapes'
SPLIT = 'val'
LEFT_ZIP = f'{CITYSCAPES_DIR}/leftImg8bit_trainvaltest.zip'
GT_ZIP   = f'{CITYSCAPES_DIR}/gtFine_trainvaltest.zip'

assert os.path.exists(LEFT_ZIP), f'Miss: {LEFT_ZIP}'
assert os.path.exists(GT_ZIP),   f'Miss: {GT_ZIP}'

# Look-up table mapping Cityscapes labelId (0..33) to trainId (0..18, 255=ignore).
id_to_trainid = np.full(256, 255, dtype=np.uint8)
for c in CSMeta.classes:
    if c.train_id not in (255, -1):
        id_to_trainid[c.id] = c.train_id


_img_resize = Resize(512, Image.BILINEAR)
_gt_resize  = Resize(512, Image.NEAREST)
_to_tensor  = ToTensor()


class CityscapesValFromZip(Dataset):
    """ Reads the pairs (image, labelIds) directly from the two zips, without extracting them"""

    def __init__(self, left_zip_path, gt_zip_path, split='val'):
        self.left_zip_path = left_zip_path
        self.gt_zip_path   = gt_zip_path

        with zipfile.ZipFile(left_zip_path) as z:
            self.img_names = sorted(
                n for n in z.namelist()
                if n.startswith(f'leftImg8bit/{split}/') and n.endswith('_leftImg8bit.png')
            )
        self.gt_names = [
            n.replace('leftImg8bit/', 'gtFine/', 1)
             .replace('_leftImg8bit.png', '_gtFine_labelIds.png')
            for n in self.img_names
        ]

        # File handle opened lazily and re-opened for each DataLoader worker.
        self._z_left = None
        self._z_gt = None

    def _ensure_open(self):
        if self._z_left is None:
            self._z_left = zipfile.ZipFile(self.left_zip_path)
            self._z_gt   = zipfile.ZipFile(self.gt_zip_path)

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        self._ensure_open()

        with self._z_left.open(self.img_names[idx]) as f:
            img = Image.open(io.BytesIO(f.read())).convert('RGB')
        with self._z_gt.open(self.gt_names[idx]) as f:
            gt_ids = Image.open(io.BytesIO(f.read()))

        img_t = _to_tensor(_img_resize(img))
        gt_ids_np = np.array(_gt_resize(gt_ids), dtype=np.uint8)

        # Map labelId to trainId via LUT; remap ignore (255) to 19 for iouEval.
        gt_train = id_to_trainid[gt_ids_np]
        gt_train[gt_train == 255] = 19
        gt_t = torch.from_numpy(gt_train.astype(np.int64)).unsqueeze(0)  # [1,H,W]

        return img_t, gt_t, self.img_names[idx], self.gt_names[idx]


cs_dataset = CityscapesValFromZip(LEFT_ZIP, GT_ZIP, split=SPLIT)
loader_cs = DataLoader(cs_dataset, batch_size=1, num_workers=0, shuffle=False)
print(f'Cityscapes {SPLIT} (from zip): {len(cs_dataset)} images')


In [ ]:
# Run inference on the Cityscapes validation set and compute mIoU.
import time

iou_eval_val = iouEval(NUM_CLASSES)
start = time.time()

with torch.no_grad():
    for step, (images, labels, _, _) in enumerate(loader_cs):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(images)
        iou_eval_val.addBatch(outputs.max(1)[1].unsqueeze(1).data, labels)
        if (step + 1) % 50 == 0:
            print(f'  {step+1}/{len(loader_cs)}')

iou_val, iou_classes = iou_eval_val.getIoU()
elapsed = time.time() - start

CLASS_NAMES = ['Road','sidewalk','building','wall','fence','pole','traffic light',
               'traffic sign','vegetation','terrain','sky','person','rider','car',
               'truck','bus','train','motorcycle','bicycle']

print(f'\nTime: {elapsed:.1f} s')
print('Per-Class IoU:')
for name, v in zip(CLASS_NAMES, iou_classes):
    print(f'  {name:14s} {float(v)*100:6.2f}')
print(f'\nMEAN IoU (ERFNet, Cityscapes {SUBSET}): {float(iou_val)*100:.2f} %')


In [ ]:
# Export per-class and mean IoU results to CSV.
import pandas as pd

miou_rows = [{'Class': name, 'IoU (%)': round(float(v)*100, 2)}
             for name, v in zip(CLASS_NAMES, iou_classes)]
miou_rows.append({'Class': 'MEAN', 'IoU (%)': round(float(iou_val)*100, 2)})

df_miou = pd.DataFrame(miou_rows)
df_miou.to_csv('step7_erfnet_miou.csv', index=False)
print('Save in: step7_erfnet_miou.csv')
df_miou
